In [ ]:
import pandas as pd
import plotly.express as px

import importlib
import sys

if "snakemake" in locals():
    spec = importlib.util.spec_from_file_location("lib", "lib/__init__.py")
else:
    spec = importlib.util.spec_from_file_location("lib", "../lib/__init__.py")
module_obj = importlib.util.module_from_spec(spec)
sys.modules["lib"] = module_obj
spec.loader.exec_module(module_obj)

from lib import PipelineConfig

In [ ]:
if "snakemake" in locals():
    pipeline_config = PipelineConfig(snakemake.params.config, **snakemake.params.pipeline_kwargs)
    analysis_name = snakemake.params.analysis_name
    analysis = pipeline_config.analyses[analysis_name]
    assert analysis.type == "routing"

    scenarios = snakemake.params.scenarios
    reference = analysis.reference
    labels = {scenario_id: item[1] for scenario_id, item in analysis.scenario_items.items()}
    sampling = pipeline_config.sampling

    assert set(scenarios.keys()) == set(analysis.scenario_items.keys())
else: # We use simulations included in the base config for testing purposes
    scenarios = dict(nofeeder="../outputs/routing/outputs/simulated_nofeeder_pt",
                     small_area_feeder="../outputs/routing/outputs/simulated_feeder_transitWithAbstractAccess_200_300_0",
                     wide_area_feeder = "../outputs/routing/outputs/simulated_feeder_transitWithAbstractAccess_400_300_0")
    reference = "nofeeder"
    labels = dict(nofeeder="baseline",
                  small_area_feeder="small area feeders",
                  wide_area_feeder="wide area feeders")
    sampling = 0.01

# Some sanity checks on the parameters
assert isinstance(scenarios, dict) and reference in scenarios
assert set(scenarios.keys()) == set(labels.keys())

In [ ]:
# We load the trips
dfs_trips = []
df_trips_per_scenario = dict()

# And the routing costs
dfs_routing_costs = []
df_routing_costs_per_scenario = dict()

for s in scenarios:
    df_routing_costs = pd.read_csv("%s/pt_routing_costs.csv" % scenarios[s], sep=";")
    df_routing_costs["scenario"] = s
    df_routing_costs_per_scenario[s] = df_routing_costs
    dfs_routing_costs.append(df_routing_costs)

    df_trips = pd.read_csv("%s/eqasim_trips.csv" % scenarios[s], sep=";")
    # We add a routing cost to trips
    df_trips = df_trips.merge(df_routing_costs.rename(columns=dict(trip_id="person_trip_id")), how="left")
    df_trips["scenario"] = s
    df_trips_per_scenario[s] = df_trips
    dfs_trips.append(df_trips)


df_trips = pd.concat(dfs_trips)
df_routing_costs = pd.concat(dfs_routing_costs)
# Just the free the memory behind the individual dfs
dfs_trips = []
dfs_routing_costs = []


In [ ]:
## Sanity checks, comparing that we compare the same set of trips
df_ref = df_trips_per_scenario[reference][["person_id", "person_trip_id", "travel_time"]]
for s in scenarios:
    if s == reference:
        continue
    df_scenario = df_trips_per_scenario[s][["person_id", "person_trip_id", "travel_time", "scenario"]]
    df_comparison = df_ref.merge(df_scenario, how="outer", indicator=True, on=["person_id", "person_trip_id"], suffixes=("_reference", "_scenario"))
    assert (df_comparison["_merge"] == "both").sum() == len(df_comparison)

# We also check that all trips have a routing cost
assert df_trips["routingCost"].isna().sum() == 0

In [ ]:
# Sanity check of the routing costs
df_ref = df_routing_costs_per_scenario[reference][["person_id", "trip_id", "routingCost"]]
for s in scenarios:
    if s == reference:
        continue
    df_scenario = df_routing_costs_per_scenario[s][["person_id", "trip_id", "routingCost"]]
    df_comparison = df_ref.merge(df_scenario, on=["person_id", "trip_id"], indicator=True, suffixes=("_reference", "_scenario"))
    assert (df_comparison["_merge"] == "both").sum() == len(df_comparison)
    # assert (df_comparison["routingCost_scenario"] > df_comparison["routingCost_reference"]).sum() == 0

# Routing cost comparison

In [ ]:
df_plot = df_routing_costs[df_routing_costs["scenario"] != reference].groupby("scenario").apply(lambda df: df_routing_costs_per_scenario[reference].drop(columns="scenario").merge(df, on=["person_id", "trip_id"], suffixes=("_reference", "_scenario")), include_groups=False).reset_index().drop(columns="level_1")
df_plot = df_plot[df_plot["routingCost_scenario"] < df_plot["routingCost_reference"]]["scenario"].value_counts()
df_plot = df_plot/sampling
df_plot = df_plot.reindex([s for s in scenarios if s!=reference]).fillna(0).reset_index()
df_plot["scenario"] = df_plot["scenario"].map(labels)
fig = px.bar(df_plot, x="scenario", y="count",
             title="Number of trips with decreased routing costs in comparison to %s" % reference,
             labels=dict(count="Number of improved trips"))
fig.show()

In [ ]:
df_plot = df_routing_costs[df_routing_costs["scenario"] != reference].groupby("scenario").apply(lambda df: df_routing_costs_per_scenario[reference].drop(columns="scenario").merge(df, on=["person_id", "trip_id"], suffixes=("_reference", "_scenario")), include_groups=False).reset_index().drop(columns="level_1")
df_plot = df_plot[df_plot["routingCost_scenario"] > df_plot["routingCost_reference"]]["scenario"].value_counts()
df_plot = df_plot/sampling
df_plot = df_plot.reindex([s for s in scenarios if s!=reference]).fillna(0).reset_index()
df_plot["scenario"] = df_plot["scenario"].map(labels)
fig = px.bar(df_plot, x="scenario", y="count",
             title="Number of trips with increased routing costs in comparison to %s" % reference,
             labels=dict(count="Number of worsened trips"))
fig.show()

# Travel time comparison
Below we simply compare the travel times for the trips to give a example of code

In [ ]:
df_comparison = df_trips.pivot_table(values="travel_time", columns="scenario", index=["person_id", "person_trip_id"]).reset_index()

In [ ]:
df_plot = df_comparison.melt(id_vars=["person_id", "person_trip_id"], value_name="travel_time", var_name="scenario").sort_values(["person_id", "person_trip_id", "scenario"])

df_plot["scenario"] = df_plot["scenario"].map(labels)
fig = px.histogram(df_plot, x="travel_time", color="scenario", barmode="overlay", title="Travel time comparison")
fig.show()

In [ ]:
for c in df_comparison.columns:
    if c in ["person_id", "person_trip_id", reference]:
        continue
    df_comparison["gain_%s" % c]  = df_comparison[reference] - df_comparison[c]

df_plot = df_comparison.melt(id_vars=["person_id", "person_trip_id"], value_vars=[c for c in df_comparison.columns if c.startswith("gain_")], value_name="gain", var_name="scenario")

df_plot["scenario"] = df_plot["scenario"].apply(lambda s: labels[s[5::]])

fig = px.histogram(df_plot[df_plot["gain"] != 0], x="gain", color="scenario", barmode="overlay")
fig.show()

# What to do next ?

Same thing as for the routing based analysis of feeder services, we compute the route costs of the trips using their components. The only difference is that here we will mainly use bar plots with the scenario on the x axis (or color) instead of line plots with the feeder radius on the x axis.

At this step it might be worth it to start factoring some code in some separate python files and importing it where needed. I am thinking about the code that computes the routing cost components for PT trips which is also used in the notebook for the feeder sensitivity analysis.

Feel free to propose other plots.